import pandas as pd
print(pd.__version__)

In [1]:
import pandas as pd
print(pd.__version__)

3.1.1


In [2]:
df = pd.read_csv("trips_lyon.csv")
print(df.shape)

(7000, 12)


In [3]:
import numpy as np
import pandas as pd

# 读取原始数据，并在副本中标准化空白。
raw = pd.read_csv("trips_lyon.csv", dtype="string", keep_default_na=False)
trips = raw.copy()
for col in trips.columns:
    trips[col] = trips[col].str.strip().replace("", pd.NA)

# 记录问题及受影响行数。各项可能涉及同一行。
audit = []

def report(column, issue, mask):
    audit.append((column, issue, int(mask.fillna(False).sum())))

for col in trips.columns:
    report(col, "缺失", trips[col].isna())

duplicate = trips.duplicated()
report("整行", "重复记录（首条之后）", duplicate)

identified = trips["trip_id"].notna() & ~duplicate
conflicts = trips.loc[identified, "trip_id"].duplicated(keep=False)
report("trip_id", "去除整行重复后，同一 ID 仍对应多条记录", conflicts)

# 校验时间和站点格式，保留无法确认的站点代码供复核。
time_text = trips["start_time"]
time_format = time_text.str.fullmatch(
    r"[0-9]{4}-[0-9]{2}-[0-9]{2} [0-9]{2}:[0-9]{2}:[0-9]{2}",
    na=False,
)
trips["start_time"] = pd.to_datetime(
    time_text.where(time_format), format="%Y-%m-%d %H:%M:%S", errors="coerce"
)
report("start_time", "格式异常或无效日期", time_text.notna() & trips["start_time"].isna())

for col in ["start_station_id", "end_station_id"]:
    bad_code = trips[col].notna() & ~trips[col].str.fullmatch(r"ST-[0-9]+", na=False)
    report(col, "站点代码格式异常", bad_code)
    trips[f"{col}_format_flag"] = bad_code

# 将无效数值置空，并单独统计各项检查结果。
numeric_cols = [
    "duration_minutes", "distance_km", "distance_meters",
    "battery_start_pct", "price_eur", "applied_rate_per_min",
]
originally_missing = trips[numeric_cols].isna()
for col in numeric_cols:
    parsed = pd.to_numeric(trips[col], errors="coerce")
    finite = parsed.notna() & ~parsed.isin([np.inf, -np.inf])
    report(col, "非数值或无穷值", trips[col].notna() & ~finite)

    invalid = parsed.lt(0)
    if col == "duration_minutes":
        invalid = parsed.le(0)
    elif col == "battery_start_pct":
        invalid = parsed.lt(0) | parsed.gt(100)

    out_of_range = finite & invalid.fillna(False)
    report(col, "超出有效范围", out_of_range)
    trips[col] = parsed.where(finite & ~out_of_range)

# 检查双单位距离。示例允许 1 米的舍入差。
km = trips["distance_km"]
meters = trips["distance_meters"]
mismatch = ((km * 1000 - meters).abs() > 1).fillna(False)
report("distance_km / distance_meters", "换算差超过 1 米", mismatch)
trips["distance_mismatch_flag"] = mismatch

# 按顺序删除重复行及缺少行程 ID 的行，并报告每次实际删除数。
clean = trips.loc[~duplicate].copy()
missing_id = clean["trip_id"].isna()
removed_missing_id = int(missing_id.sum())
clean = clean.loc[~missing_id].copy()
clean["trip_id_conflict_flag"] = clean["trip_id"].duplicated(keep=False)

# 从有效的另一列补齐原本缺失的距离。
fill_km = originally_missing.loc[clean.index, "distance_km"]
fill_meters = originally_missing.loc[clean.index, "distance_meters"]
clean.loc[fill_km, "distance_km"] = clean.loc[fill_km, "distance_meters"] / 1000
clean.loc[fill_meters, "distance_meters"] = clean.loc[fill_meters, "distance_km"] * 1000

# 用 1.5 倍 IQR 标记异常。距离阈值排除两列距离不一致的记录。
for col in ["duration_minutes", "distance_km"]:
    eligible = clean[col].notna()
    if col == "distance_km":
        eligible &= ~clean["distance_mismatch_flag"]

    sample = clean.loc[eligible, col]
    flag = pd.Series(False, index=clean.index)
    if len(sample) >= 4:
        q1, q3 = sample.quantile([0.25, 0.75])
        iqr = q3 - q1
        if iqr > 0:
            flag.loc[sample.index] = (
                sample.lt(q1 - 1.5 * iqr) | sample.gt(q3 + 1.5 * iqr)
            )
    clean[f"{col}_outlier_flag"] = flag
    report(col, "IQR 异常（基于保留行）", flag)

# 输出审计表、行数核对和清洗文件。
print(pd.DataFrame(audit, columns=["字段", "问题", "受影响行数"]).to_string(index=False))
print(f"原始行数 = {len(raw)}")
print(f"删除重复行 = {int(duplicate.sum())}")
print(f"删除缺少 trip_id 的行（去重后） = {removed_missing_id}")
print(f"最终行数 = {len(clean)}")
assert len(clean) == len(raw) - int(duplicate.sum()) - removed_missing_id
clean.to_csv("trips_lyon_clean.csv", index=False)

                           字段                    问题  受影响行数
                      trip_id                    缺失      0
                      user_id                    缺失      0
                   start_time                    缺失      0
             start_station_id                    缺失      0
               end_station_id                    缺失    407
                 vehicle_type                    缺失      0
             duration_minutes                    缺失      0
                  distance_km                    缺失      0
              distance_meters                    缺失      0
            battery_start_pct                    缺失      0
                    price_eur                    缺失      0
         applied_rate_per_min                    缺失      0
                           整行            重复记录（首条之后）      0
                      trip_id 去除整行重复后，同一 ID 仍对应多条记录      0
                   start_time             格式异常或无效日期      0
             start_station_id              站点代码格式异常     

In [5]:
import pandas as pd
import numpy as np

# 1. Load raw data and inspect total row and column counts without modifying the original DataFrame
df_raw = pd.read_csv("trips_lyon.csv")
print(f"Raw dataset shape: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns")

# 2. Audit all columns before cleaning and record data quality issues
audit_issues = []
audit_issues.append(("trip_id", "Missing or duplicate trip_id", df_raw["trip_id"].isna().sum() + df_raw["trip_id"].duplicated().sum()))
audit_issues.append(("user_id", "Missing user_id", df_raw["user_id"].isna().sum()))
parsed_time = pd.to_datetime(df_raw["start_time"], format="%Y-%m-%d %H:%M:%S", errors="coerce")
audit_issues.append(("start_time", "Missing or invalid timestamp format", parsed_time.isna().sum()))
valid_start_st = df_raw["start_station_id"].astype(str).str.match(r"^ST-\d+$", na=False)
valid_end_st = df_raw["end_station_id"].astype(str).str.match(r"^ST-\d+$", na=False)
audit_issues.append(("start_station_id", "Missing or invalid station format (expected ST-XXX)", (~valid_start_st).sum()))
audit_issues.append(("end_station_id", "Missing or invalid station format (expected ST-XXX)", (~valid_end_st).sum()))
audit_issues.append(("vehicle_type", "Missing or unstandardized casing/whitespace", df_raw["vehicle_type"].isna().sum() + (df_raw["vehicle_type"].dropna() != df_raw["vehicle_type"].dropna().str.strip().str.lower()).sum()))
audit_issues.append(("duration_minutes", "Missing, non-positive (<= 0), or extreme outliers (> 1440 min)", (df_raw["duration_minutes"].isna() | (df_raw["duration_minutes"] <= 0) | (df_raw["duration_minutes"] > 1440)).sum()))
audit_issues.append(("distance_km / distance_meters", "Missing, negative (< 0), or unit mismatch (> 0.05 km diff)", (df_raw["distance_km"].isna() | df_raw["distance_meters"].isna() | (df_raw["distance_km"] < 0) | (df_raw["distance_meters"] < 0) | ((df_raw["distance_km"] - df_raw["distance_meters"] / 1000.0).abs() > 0.05)).sum()))
audit_issues.append(("battery_start_pct", "Missing or out of valid range [0, 100]", (df_raw["battery_start_pct"].isna() | (df_raw["battery_start_pct"] < 0) | (df_raw["battery_start_pct"] > 100)).sum()))
audit_issues.append(("price_eur / applied_rate_per_min", "Missing or negative pricing values (< 0)", (df_raw["price_eur"].isna() | df_raw["applied_rate_per_min"].isna() | (df_raw["price_eur"] < 0) | (df_raw["applied_rate_per_min"] < 0)).sum()))

print("\n--- (1) Problems Found (Pre-Cleaning Audit) ---")
for idx, (col, desc, count) in enumerate(audit_issues, 1):
    print(f"{idx}. Column: {col} | Issue: {desc} | Rows affected: {count}")

# 3. Create a working copy so the raw DataFrame remains untouched
df_clean = df_raw.copy()
removal_log = []

# 4. Report and drop rows with missing critical identifiers or duplicate trip_id
before = len(df_clean)
missing_ids = df_clean["trip_id"].isna() | df_clean["user_id"].isna()
print(f"\n[Cleaning Step] Missing trip_id or user_id count: {missing_ids.sum()}")
df_clean = df_clean[~missing_ids].drop_duplicates(subset=["trip_id"], keep="first")
removed = before - len(df_clean)
removal_log.append(("Missing trip_id/user_id or duplicate trip_id", removed))
print(f"Removed {removed} rows -> Reason: Primary/foreign keys cannot be null or duplicated. Remaining: {len(df_clean)}")

# 5. Parse start_time to datetime and remove rows with unparseable timestamps
before = len(df_clean)
df_clean["start_time"] = pd.to_datetime(df_clean["start_time"], format="%Y-%m-%d %H:%M:%S", errors="coerce")
invalid_time = df_clean["start_time"].isna()
print(f"[Cleaning Step] Invalid/missing start_time count: {invalid_time.sum()}")
df_clean = df_clean[~invalid_time]
removed = before - len(df_clean)
removal_log.append(("Missing or invalid start_time format", removed))
print(f"Removed {removed} rows -> Reason: Invalid timestamp format (expected YYYY-MM-DD HH:MM:SS). Remaining: {len(df_clean)}")

# 6. Validate station ID format (ST-XXX) and drop rows with missing or malformed station codes
before = len(df_clean)
df_clean["start_station_id"] = df_clean["start_station_id"].astype(str).str.strip().str.upper()
df_clean["end_station_id"] = df_clean["end_station_id"].astype(str).str.strip().str.upper()
valid_stations = df_clean["start_station_id"].str.match(r"^ST-\d+$") & df_clean["end_station_id"].str.match(r"^ST-\d+$")
df_clean = df_clean[valid_stations]
removed = before - len(df_clean)
removal_log.append(("Invalid or missing start_station_id / end_station_id", removed))
print(f"Removed {removed} rows -> Reason: Station IDs must match format ST-XXX to link with stations_metadata.csv. Remaining: {len(df_clean)}")

# 7. Standardize vehicle_type text and drop rows where vehicle_type is missing
before = len(df_clean)
missing_vehicle = df_clean["vehicle_type"].isna()
print(f"[Cleaning Step] Missing vehicle_type count: {missing_vehicle.sum()}")
df_clean = df_clean[~missing_vehicle]
df_clean["vehicle_type"] = df_clean["vehicle_type"].astype(str).str.strip().str.lower()
removed = before - len(df_clean)
removal_log.append(("Missing vehicle_type", removed))
print(f"Removed {removed} rows -> Reason: Vehicle type is required to analyze bike vs. scooter usage and pricing. Remaining: {len(df_clean)}")

# 8. Reconcile distance_km and distance_meters, drop negative values, and reconcile unit mismatches
before = len(df_clean)
both_dist_missing = df_clean["distance_km"].isna() & df_clean["distance_meters"].isna()
print(f"[Cleaning Step] Both distance_km and distance_meters missing: {both_dist_missing.sum()}")
df_clean = df_clean[~both_dist_missing]
df_clean["distance_km"] = df_clean["distance_km"].fillna(df_clean["distance_meters"] / 1000.0)
df_clean["distance_meters"] = df_clean["distance_meters"].fillna(df_clean["distance_km"] * 1000.0)
invalid_dist = (df_clean["distance_km"] < 0) | (df_clean["distance_meters"] < 0) | ((df_clean["distance_km"] - df_clean["distance_meters"] / 1000.0).abs() > 0.05)
df_clean = df_clean[~invalid_dist]
removed = before - len(df_clean)
removal_log.append(("Missing, negative, or conflicting distance_km & distance_meters", removed))
print(f"Removed {removed} rows -> Reason: Negative distances or irreconcilable km/meter discrepancies (> 50m). Remaining: {len(df_clean)}")

# 9. Remove missing or out-of-range battery_start_pct values outside [0, 100]
before = len(df_clean)
invalid_battery = df_clean["battery_start_pct"].isna() | (df_clean["battery_start_pct"] < 0) | (df_clean["battery_start_pct"] > 100)
print(f"[Cleaning Step] Missing or out-of-range battery_start_pct count: {invalid_battery.sum()}")
df_clean = df_clean[~invalid_battery]
removed = before - len(df_clean)
removal_log.append(("Missing or invalid battery_start_pct (<0 or >100)", removed))
print(f"Removed {removed} rows -> Reason: Battery percentage must be within [0, 100]. Remaining: {len(df_clean)}")

# 10. Remove missing, non-positive, or extreme duration_minutes and negative pricing records
before = len(df_clean)
invalid_duration_price = (
    df_clean["duration_minutes"].isna() | (df_clean["duration_minutes"] <= 0) |
    df_clean["price_eur"].isna() | (df_clean["price_eur"] < 0) |
    df_clean["applied_rate_per_min"].isna() | (df_clean["applied_rate_per_min"] < 0)
)
print(f"[Cleaning Step] Invalid duration_minutes (<=0/NaN) or negative/NaN pricing count: {invalid_duration_price.sum()}")
df_clean = df_clean[~invalid_duration_price]
removed = before - len(df_clean)
removal_log.append(("Non-positive duration_minutes or negative/missing price/rate", removed))
print(f"Removed {removed} rows -> Reason: Duration must be > 0 and pricing fields must be non-negative and non-null. Remaining: {len(df_clean)}")

# 11. Detect and remove statistical outliers in duration_minutes, distance_km, and price_eur using IQR per vehicle_type
before = len(df_clean)
outlier_mask = pd.Series(False, index=df_clean.index)
for col in ["duration_minutes", "distance_km", "price_eur"]:
    q1 = df_clean.groupby("vehicle_type")[col].transform(lambda s: s.quantile(0.25))
    q3 = df_clean.groupby("vehicle_type")[col].transform(lambda s: s.quantile(0.75))
    iqr = q3 - q1
    lower_bound = q1 - 3.0 * iqr
    upper_bound = q3 + 3.0 * iqr
    outlier_mask = outlier_mask | (df_clean[col] < lower_bound) | (df_clean[col] > upper_bound)

df_clean = df_clean[~outlier_mask]
removed = before - len(df_clean)
removal_log.append(("Extreme outliers (3*IQR by vehicle_type) in duration, distance, or price", removed))
print(f"Removed {removed} rows -> Reason: Extreme statistical outliers distort pricing model calibration. Remaining: {len(df_clean)}")

# 12. Export cleaned dataset and print final results summary
df_clean.to_csv("trips_lyon_clean.csv", index=False)

print("\n--- (3) Results Summary ---")
print(f"Raw rows: {len(df_raw)}")
for rule, count in removal_log:
    print(f" - Removed ({rule}): {count}")
print(f"Total rows removed: {len(df_raw) - len(df_clean)}")
print(f"Final row count: {len(df_clean)}")

Raw dataset shape: 7000 rows, 12 columns

--- (1) Problems Found (Pre-Cleaning Audit) ---
1. Column: trip_id | Issue: Missing or duplicate trip_id | Rows affected: 0
2. Column: user_id | Issue: Missing user_id | Rows affected: 0
3. Column: start_time | Issue: Missing or invalid timestamp format | Rows affected: 0
4. Column: start_station_id | Issue: Missing or invalid station format (expected ST-XXX) | Rows affected: 0
5. Column: end_station_id | Issue: Missing or invalid station format (expected ST-XXX) | Rows affected: 407
6. Column: vehicle_type | Issue: Missing or unstandardized casing/whitespace | Rows affected: 2257
7. Column: duration_minutes | Issue: Missing, non-positive (<= 0), or extreme outliers (> 1440 min) | Rows affected: 205
8. Column: distance_km / distance_meters | Issue: Missing, negative (< 0), or unit mismatch (> 0.05 km diff) | Rows affected: 0
9. Column: battery_start_pct | Issue: Missing or out of valid range [0, 100] | Rows affected: 0
10. Column: price_eur / a

In [6]:
import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# 0. Load dataset & verify schema (only the 12 specified columns)
# ---------------------------------------------------------------------------
EXPECTED_COLS = [
    "trip_id",
    "user_id",
    "start_time",
    "start_station_id",
    "end_station_id",
    "vehicle_type",
    "duration_minutes",
    "distance_km",
    "distance_meters",
    "battery_start_pct",
    "price_eur",
    "applied_rate_per_min",
]

df_raw = pd.read_csv("trips_lyon.csv", usecols=EXPECTED_COLS)
initial_rows = len(df_raw)
df = df_raw.copy()

print(f"Loaded trips_lyon.csv | Initial row count: {initial_rows:,}\n")


def log_step(step_name, found_desc, affected_count, action_desc, reason_desc, current_df):
    """Standardized audit logger so no change occurs silently."""
    print(f"=== {step_name} ===")
    print(f"  Found         : {found_desc}")
    print(f"  Affected rows : {affected_count:,}")
    print(f"  Action        : {action_desc}")
    print(f"  Why           : {reason_desc}")
    print(f"  Row count now : {len(current_df):,}\n")


# ---------------------------------------------------------------------------
# Step 1: Audit & remove duplicate trip_id rows
# Assumption: trip_id is a unique identifier; keep first occurrence.
# ---------------------------------------------------------------------------
dup_mask = df.duplicated(subset=["trip_id"], keep="first")
dup_count = int(dup_mask.sum())
df = df.loc[~dup_mask].copy()
log_step(
    "Step 1: Duplicate trip_id check",
    f"{dup_count} duplicate trip_id records detected",
    dup_count,
    "Dropped duplicate rows keeping first occurrence" if dup_count else "No rows dropped",
    "Each row must represent one unique trip rental (trip_id is unique).",
    df,
)

# ---------------------------------------------------------------------------
# Step 2 (Rule 1): Filter valid trip durations (0 < duration_minutes < 1440)
# ---------------------------------------------------------------------------
df["duration_minutes"] = pd.to_numeric(df["duration_minutes"], errors="coerce")
invalid_dur_mask = (
    df["duration_minutes"].isna()
    | (df["duration_minutes"] <= 0)
    | (df["duration_minutes"] >= 1440)
)
invalid_dur_count = int(invalid_dur_mask.sum())
df = df.loc[~invalid_dur_mask].copy()
log_step(
    "Step 2 (Rule 1): Valid duration filter",
    f"{invalid_dur_count} rows with null, <= 0, or >= 1440 duration_minutes",
    invalid_dur_count,
    "Removed rows outside (0, 1440) minutes",
    "Rule 1 defines valid trips strictly as 0 < duration_minutes < 1440 (< 24h).",
    df,
)

# ---------------------------------------------------------------------------
# Step 3 (Rule 2): Standardize vehicle_type into 3 canonical labels
# ---------------------------------------------------------------------------
raw_vt = df["vehicle_type"].astype("string").str.strip().str.lower()
# Normalize separators (spaces/underscores/hyphens) for reliable mapping
norm_vt = raw_vt.str.replace(r"[\s_]+", "-", regex=True)

vehicle_map = {
    "e-bike": "e-bike",
    "ebike": "e-bike",
    "electric-bike": "e-bike",
    "electric-bicycle": "e-bike",
    "velo-electrique": "e-bike",
    "scooter": "scooter",
    "e-scooter": "scooter",
    "escooter": "scooter",
    "kick-scooter": "scooter",
    "trottinette": "scooter",
    "mechanical-bike": "mechanical_bike",
    "mechanicalbike": "mechanical_bike",
    "mech-bike": "mechanical_bike",
    "bike": "mechanical_bike",
    "classic-bike": "mechanical_bike",
    "regular-bike": "mechanical_bike",
    "velo-mecanique": "mechanical_bike",
}

mapped_vt = norm_vt.map(vehicle_map)
changed_vt_count = int((df["vehicle_type"].astype("string") != mapped_vt).fillna(True).sum())
unmapped_mask = mapped_vt.isna()
unmapped_count = int(unmapped_mask.sum())

if unmapped_count > 0:
    print("Unmapped vehicle_type values found:", df.loc[unmapped_mask, "vehicle_type"].unique())

df["vehicle_type"] = mapped_vt
# Assumption: drop rows with unidentifiable vehicle_type if any remain after mapping
df = df.loc[~unmapped_mask].copy()

log_step(
    "Step 3 (Rule 2): Standardize vehicle_type",
    f"{changed_vt_count} rows with non-canonical spelling/casing; {unmapped_count} unmappable",
    changed_vt_count,
    "Unified variants into {e-bike, scooter, mechanical_bike} and dropped unmappable rows",
    "Rule 2 requires exact canonical labels for fleet segmentation.",
    df,
)
print("vehicle_type value_counts() after standardization:")
print(df["vehicle_type"].value_counts(dropna=False), "\n")

# ---------------------------------------------------------------------------
# Step 4 (Rule 3): Flag outside_station without dropping or filling end_station_id
# ---------------------------------------------------------------------------
# Treat empty/whitespace-only strings as missing (NaN) while preserving NaN
df["end_station_id"] = df["end_station_id"].replace(r"^\s*$", np.nan, regex=True)
df["outside_station"] = df["end_station_id"].isna()
outside_count = int(df["outside_station"].sum())

log_step(
    "Step 4 (Rule 3): Flag dockless trips (outside_station)",
    f"{outside_count} trips with empty/null end_station_id",
    outside_count,
    "Added boolean column 'outside_station' (True when end_station_id is empty); kept all rows and left end_station_id unfilled",
    "Rule 3 specifies empty end_station_id means left outside a station and must never be dropped or filled.",
    df,
)

# ---------------------------------------------------------------------------
# Step 5: Parse start_time & check start_station_id
# Assumption: Drop rows with invalid/missing start_time since temporal analysis requires it.
# ---------------------------------------------------------------------------
parsed_start = pd.to_datetime(df["start_time"], format="%Y-%m-%d %H:%M:%S", errors="coerce")
bad_time_mask = parsed_start.isna()
bad_time_count = int(bad_time_mask.sum())
df["start_time"] = parsed_start
df = df.loc[~bad_time_mask].copy()

log_step(
    "Step 5: Parse start_time timestamps",
    f"{bad_time_count} rows with null or malformed start_time",
    bad_time_count,
    "Converted start_time to datetime64 and removed unparseable rows",
    "Valid timestamps are required to answer when the fleet is used.",
    df,
)

# ---------------------------------------------------------------------------
# Step 6: Reconcile distance_km and distance_meters
# Assumption: Keep both schema columns; backfill missing from counterpart; flag mismatches/negatives.
# ---------------------------------------------------------------------------
for col in ["distance_km", "distance_meters"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

neg_dist_mask = (df["distance_km"] < 0) | (df["distance_meters"] < 0)
neg_dist_count = int(neg_dist_mask.sum())
df = df.loc[~neg_dist_mask].copy()

km_from_m = df["distance_km"].isna() & df["distance_meters"].notna()
m_from_km = df["distance_meters"].isna() & df["distance_km"].notna()
df.loc[km_from_m, "distance_km"] = df.loc[km_from_m, "distance_meters"] / 1000.0
df.loc[m_from_km, "distance_meters"] = df.loc[m_from_km, "distance_km"] * 1000.0

unit_mismatch_mask = (
    df["distance_km"].notna()
    & df["distance_meters"].notna()
    & ~np.isclose(df["distance_meters"], df["distance_km"] * 1000.0, atol=1.0)
)
mismatch_count = int(unit_mismatch_mask.sum())

log_step(
    "Step 6: Audit distance_km vs distance_meters",
    f"{neg_dist_count} negative distances; {int(km_from_m.sum() + m_from_km.sum())} backfilled; {mismatch_count} unit mismatches (>1m diff)",
    neg_dist_count + int(km_from_m.sum() + m_from_km.sum()) + mismatch_count,
    "Dropped negative distances; cross-filled missing unit pairs; retained mismatches for review",
    "Distances cannot be negative, and unit consistency ensures reliable distance features.",
    df,
)

# ---------------------------------------------------------------------------
# Step 7: Validate battery_start_pct (0 to 100) and pricing columns
# Assumption: Out-of-range battery values -> NaN; negative prices/rates -> drop.
# ---------------------------------------------------------------------------
df["battery_start_pct"] = pd.to_numeric(df["battery_start_pct"], errors="coerce")
bad_batt_mask = df["battery_start_pct"].notna() & (
    (df["battery_start_pct"] < 0) | (df["battery_start_pct"] > 100)
)
bad_batt_count = int(bad_batt_mask.sum())
df.loc[bad_batt_mask, "battery_start_pct"] = np.nan

log_step(
    "Step 7a: Validate battery_start_pct domain [0, 100]",
    f"{bad_batt_count} rows outside [0, 100]%",
    bad_batt_count,
    "Replaced out-of-bounds battery_start_pct values with NaN (kept rows)",
    "Battery percentage must lie within 0–100; nulling avoids losing valid trip/price records.",
    df,
)

for col in ["price_eur", "applied_rate_per_min"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

bad_price_mask = (
    df["price_eur"].isna()
    | (df["price_eur"] < 0)
    | df["applied_rate_per_min"].isna()
    | (df["applied_rate_per_min"] < 0)
)
bad_price_count = int(bad_price_mask.sum())
df = df.loc[~bad_price_mask].copy()

log_step(
    "Step 7b: Validate price_eur and applied_rate_per_min",
    f"{bad_price_count} rows with null or negative price_eur / applied_rate_per_min",
    bad_price_count,
    "Removed rows with missing or negative pricing values",
    "Clean non-negative pricing fields are required to model trip price drivers.",
    df,
)

# ---------------------------------------------------------------------------
# Step 8 (Rule 4): Flag duration_minutes outliers via IQR (k = 1.5)
# ---------------------------------------------------------------------------
q1 = df["duration_minutes"].quantile(0.25)
q3 = df["duration_minutes"].quantile(0.75)
iqr = q3 - q1
lower_fence = q1 - 1.5 * iqr
upper_fence = q3 + 1.5 * iqr

df["duration_outlier"] = (df["duration_minutes"] < lower_fence) | (
    df["duration_minutes"] > upper_fence
)
outlier_count = int(df["duration_outlier"].sum())

log_step(
    "Step 8 (Rule 4): Flag duration_minutes IQR outliers (k = 1.5)",
    f"Q1={q1:.2f}, Q3={q3:.2f}, IQR={iqr:.2f}, Fences=[{lower_fence:.2f}, {upper_fence:.2f}] -> {outlier_count} outliers",
    outlier_count,
    "Added boolean column 'duration_outlier'; retained all outlier rows",
    "Rule 4 requires flagging IQR outliers on cleaned duration_minutes without deleting them.",
    df,
)

# ---------------------------------------------------------------------------
# Step 9: Final Before / After Summary & Export
# ---------------------------------------------------------------------------
final_rows = len(df)
removed_rows = initial_rows - final_rows
pct_retained = (final_rows / initial_rows * 100) if initial_rows else 0.0

print("==========================================")
print("BEFORE / AFTER ROW-COUNT SUMMARY")
print("==========================================")
print(f"Initial rows (raw)      : {initial_rows:,}")
print(f"Rows removed            : {removed_rows:,}")
print(f"Final rows (cleaned)    : {final_rows:,} ({pct_retained:.2f}% retained)")
print(f"Outside-station trips   : {int(df['outside_station'].sum()):,} (kept)")
print(f"Duration IQR outliers   : {int(df['duration_outlier'].sum()):,} (kept & flagged)")
print("==========================================")

df.to_csv("trips_lyon_cleaned.csv", index=False)

Loaded trips_lyon.csv | Initial row count: 7,000

=== Step 1: Duplicate trip_id check ===
  Found         : 0 duplicate trip_id records detected
  Affected rows : 0
  Action        : No rows dropped
  Why           : Each row must represent one unique trip rental (trip_id is unique).
  Row count now : 7,000

=== Step 2 (Rule 1): Valid duration filter ===
  Found         : 205 rows with null, <= 0, or >= 1440 duration_minutes
  Affected rows : 205
  Action        : Removed rows outside (0, 1440) minutes
  Why           : Rule 1 defines valid trips strictly as 0 < duration_minutes < 1440 (< 24h).
  Row count now : 6,795

=== Step 3 (Rule 2): Standardize vehicle_type ===
  Found         : 3367 rows with non-canonical spelling/casing; 0 unmappable
  Affected rows : 3,367
  Action        : Unified variants into {e-bike, scooter, mechanical_bike} and dropped unmappable rows
  Why           : Rule 2 requires exact canonical labels for fleet segmentation.
  Row count now : 6,795

vehicle_type 

In [7]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd


def clean_trips(source):
    trips = pd.read_csv(source, dtype="string", keep_default_na=False)
    trips = trips.apply(lambda col: col.str.strip().replace("", pd.NA))
    print(f"Input: {len(trips):,} rows")
    print("Blank values by column:\n", trips.isna().sum().to_string())
    print(f"Exact duplicate rows beyond the first: {trips.duplicated().sum():,}")
    print(f"Repeated trip IDs beyond the first: {trips.trip_id.dropna().duplicated().sum():,}")

    duration = pd.to_numeric(trips.duration_minutes, errors="coerce")
    finite = np.isfinite(duration).fillna(False)
    valid = finite & duration.gt(0).fillna(False) & duration.lt(1440).fillna(False)
    print(f"Duration at or below zero: {(finite & duration.le(0)).sum():,}")
    print(f"Duration at or above 24 hours: {(finite & duration.ge(1440)).sum():,}")
    print(f"Missing, nonnumeric or nonfinite duration: {(~finite).sum():,}")
    print(f"Removing {(~valid).sum():,} rows outside the assumed valid duration range")
    trips = trips.loc[valid].copy()
    trips["duration_minutes"] = duration.loc[valid]
    print(f"Rows after duration filter: {len(trips):,}")

    aliases = {"electric_bike": "e-bike", "trottinette": "scooter"}
    vehicle = trips.vehicle_type.str.casefold().replace(aliases)
    known = vehicle.isin(["e-bike", "scooter", "mechanical_bike"])
    print(f"Unrecognized or missing vehicle labels: {(~known).sum():,}")
    changed = known & vehicle.ne(trips.vehicle_type).fillna(False)
    trips.loc[changed, "vehicle_type"] = vehicle.loc[changed]
    print(f"Standardized {changed.sum():,} known aliases and case variants")
    print(trips.vehicle_type.value_counts(dropna=False).to_string())

    trips["outside_station"] = trips.end_station_id.isna()
    print(f"Blank end stations retained and flagged: {trips.outside_station.sum():,}")

    trips["start_time"] = pd.to_datetime(
        trips.start_time, format="%Y-%m-%d %H:%M:%S", errors="raise"
    )
    for col in ["distance_km", "distance_meters", "battery_start_pct",
                "price_eur", "applied_rate_per_min"]:
        trips[col] = pd.to_numeric(trips[col], errors="raise")

    print("Distance unit disagreements (> 0.01 m):",
          ((trips.distance_km * 1000 - trips.distance_meters).abs() > 0.01).sum())
    print("Battery readings outside 0–100:",
          (trips.battery_start_pct.lt(0) | trips.battery_start_pct.gt(100)).sum())

    for col in ["duration_minutes", "distance_km", "price_eur"]:
        values = trips[col]
        q1, q3 = values.quantile([0.25, 0.75])
        spread = 1.5 * (q3 - q1)
        lower, upper = q1 - spread, q3 + spread
        low, high = values.lt(lower), values.gt(upper)
        trips[f"{col}_outlier"] = (low | high).fillna(False)
        print(f"{col} IQR fences [{lower:.4f}, {upper:.4f}]: "
              f"{low.sum():,} below, {high.sum():,} above, all retained")

    print(f"Rows with any outlier flag: {trips.filter(like='_outlier').any(axis=1).sum():,}")
    print(f"Final rows: {len(trips):,}")
    return trips.reset_index(drop=True)


if __name__ == "__main__":
    if len(sys.argv) not in (2, 3):
        raise SystemExit("Usage: python clean_trips_lyon.py INPUT.csv [OUTPUT.csv]")
    source = Path(sys.argv[1])
    cleaned = clean_trips(source)
    if len(sys.argv) == 3:
        with Path(sys.argv[2]).open("x", encoding="utf-8", newline="") as output:
            cleaned.to_csv(output, index=False)



FileNotFoundError: [Errno 2] No such file or directory: '--f=/home/codespace/.local/share/jupyter/runtime/kernel-v3b5edbe9ce1c2f5da6823f91bd54ac1d4aec8043d.json'

In [8]:

import pandas as pd
import numpy as np

# ==========================================
# 1. Load data
# ==========================================

# Change this path to the CSV file you choose.
file_path = "trips_lyon.csv"

df = pd.read_csv(file_path)
original_rows = len(df)

print("=== INITIAL DATA ===")
print(f"Original rows: {original_rows}")
print(f"Original columns: {len(df.columns)}")
print("\nMissing values:")
print(df.isna().sum())

# Keep an audit log
audit = []

def log_step(step, affected, action, reason):
    audit.append({
        "Step": step,
        "Affected rows": affected,
        "Action": action,
        "Reason": reason,
        "Rows remaining": len(df)
    })
    print(f"\n--- {step} ---")
    print(f"Affected rows: {affected}")
    print(f"Action: {action}")
    print(f"Reason: {reason}")
    print(f"Rows remaining: {len(df)}")


# ==========================================
# 2. Validate trip duration
# ==========================================

df["duration_minutes"] = pd.to_numeric(
    df["duration_minutes"], errors="coerce"
)

invalid_duration = (
    df["duration_minutes"].isna()
    | (df["duration_minutes"] <= 0)
    | (df["duration_minutes"] >= 1440)
)

invalid_count = int(invalid_duration.sum())

df = df.loc[~invalid_duration].copy()

log_step(
    "Invalid trip duration",
    invalid_count,
    "Removed invalid trips",
    "Valid duration must be > 0 and < 1440 minutes"
)


# ==========================================
# 3. Standardize vehicle types
# ==========================================

print("\nOriginal vehicle types:")
print(df["vehicle_type"].value_counts(dropna=False))

original_types = df["vehicle_type"].copy()

# Normalize spelling, case and separators
normalized = (
    df["vehicle_type"]
    .astype("string")
    .str.strip()
    .str.lower()
    .str.replace(r"[\s_-]+", "", regex=True)
)

vehicle_mapping = {
    "ebike": "e-bike",
    "electricbike": "e-bike",
    "electricbicycle": "e-bike",
    "scooter": "scooter",
    "electricscooter": "scooter",
    "escooter": "scooter",
    "mechanicalbike": "mechanical_bike",
    "mechanicalbicycle": "mechanical_bike",
    "regularbike": "mechanical_bike",
    "normalbike": "mechanical_bike"
}

df["vehicle_type"] = normalized.map(vehicle_mapping)

changed_types = int((
    original_types.astype("string")
    .ne(df["vehicle_type"])
    .fillna(False)
).sum())

unknown_types = int(df["vehicle_type"].isna().sum())

log_step(
    "Vehicle type standardization",
    changed_types,
    "Standardized recognized labels; unknown labels flagged",
    "Only three canonical vehicle types are allowed"
)

print("\nStandardized vehicle types:")
print(df["vehicle_type"].value_counts(dropna=False))
print(f"Unknown or missing types: {unknown_types}")


# ==========================================
# 4. Identify trips outside stations
# ==========================================

end_station = df["end_station_id"].astype("string")

df["outside_station"] = (
    end_station.isna()
    | end_station.str.strip().eq("").fillna(False)
)

outside_count = int(df["outside_station"].sum())

log_step(
    "Outside-station trips",
    outside_count,
    "Kept all rows; added outside_station flag",
    "Missing end station indicates parking outside a station"
)


# ==========================================
# 5. Detect duration outliers (IQR)
# ==========================================

Q1 = df["duration_minutes"].quantile(0.25)
Q3 = df["duration_minutes"].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

df["duration_outlier"] = (
    (df["duration_minutes"] < lower_bound)
    | (df["duration_minutes"] > upper_bound)
)

outlier_count = int(df["duration_outlier"].sum())

print(f"\nIQR lower fence: {lower_bound:.2f}")
print(f"IQR upper fence: {upper_bound:.2f}")

log_step(
    "Duration outliers",
    outlier_count,
    "Flagged outliers; no rows removed",
    "IQR method with k = 1.5"
)


# ==========================================
# 6. Additional data quality checks
# ==========================================

# Duplicate trip IDs
duplicate_ids = int(
    df["trip_id"].duplicated(keep=False).sum()
)

log_step(
    "Duplicate trip IDs",
    duplicate_ids,
    "Reported only",
    "Trip IDs should be unique; no automatic deletion"
)

# Invalid start times
parsed_time = pd.to_datetime(
    df["start_time"],
    format="%Y-%m-%d %H:%M:%S",
    errors="coerce"
)

invalid_times = int(parsed_time.isna().sum())

log_step(
    "Invalid start times",
    invalid_times,
    "Reported only",
    "Start times must follow YYYY-MM-DD HH:MM:SS"
)

# Distance consistency
distance_km = pd.to_numeric(
    df["distance_km"], errors="coerce"
)

distance_meters = pd.to_numeric(
    df["distance_meters"], errors="coerce"
)

distance_mismatch = (
    distance_km.notna()
    & distance_meters.notna()
    & ~np.isclose(
        distance_km * 1000,
        distance_meters,
        atol=1,
        rtol=0
    )
)

log_step(
    "Distance consistency",
    int(distance_mismatch.sum()),
    "Reported only",
    "Kilometres and metres should agree within 1 metre"
)

# Battery percentage
battery = pd.to_numeric(
    df["battery_start_pct"], errors="coerce"
)

invalid_battery = (
    battery.isna()
    | (battery < 0)
    | (battery > 100)
)

log_step(
    "Invalid battery percentage",
    int(invalid_battery.sum()),
    "Reported only",
    "Battery percentage must be between 0 and 100"
)

# Negative or missing prices and rates
price = pd.to_numeric(
    df["price_eur"], errors="coerce"
)

rate = pd.to_numeric(
    df["applied_rate_per_min"], errors="coerce"
)

invalid_price = (
    price.isna()
    | rate.isna()
    | (price < 0)
    | (rate < 0)
)

log_step(
    "Invalid prices or rates",
    int(invalid_price.sum()),
    "Reported only",
    "Prices and rates must be non-negative"
)

# Price consistency (diagnostic only)
# Assumption: simple per-minute pricing, no fixed fees.
price_mismatch = (
    price.notna()
    & rate.notna()
    & (price >= 0)
    & (rate >= 0)
    & ~np.isclose(
        price,
        df["duration_minutes"] * rate,
        atol=0.02,
        rtol=0
    )
)

log_step(
    "Price calculation differences",
    int(price_mismatch.sum()),
    "Reported only",
    "Assumes price = duration x rate; other fees may apply"
)


# ==========================================
# 7. Final audit summary
# ==========================================

final_rows = len(df)

print("\n========== FINAL SUMMARY ==========")
print(f"Original rows: {original_rows}")
print(f"Final rows: {final_rows}")
print(f"Rows removed: {original_rows - final_rows}")
print(f"Rows retained: {final_rows / original_rows:.2%}"
      if original_rows else "Rows retained: N/A")

print("\nAudit report:")
audit_df = pd.DataFrame(audit)
print(audit_df.to_string(index=False))

print("\nFinal vehicle distribution:")
print(df["vehicle_type"].value_counts(dropna=False))

print("\nFinal dataset shape:")
print(df.shape)


# ==========================================
# 8. Export cleaned data and audit report
# ==========================================

df.to_csv(
    "trips_lyon_cleaned.csv",
    index=False,
    encoding="utf-8-sig"
)

audit_df.to_csv(
    "trips_lyon_audit_report.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nFiles saved:")
print("trips_lyon_cleaned.csv")
print("trips_lyon_audit_report.csv")


=== INITIAL DATA ===
Original rows: 7000
Original columns: 12

Missing values:
trip_id                   0
user_id                   0
start_time                0
start_station_id          0
end_station_id          407
vehicle_type              0
duration_minutes          0
distance_km               0
distance_meters           0
battery_start_pct         0
price_eur                 0
applied_rate_per_min      0
dtype: int64

--- Invalid trip duration ---
Affected rows: 205
Action: Removed invalid trips
Reason: Valid duration must be > 0 and < 1440 minutes
Rows remaining: 6795

Original vehicle types:
vehicle_type
electric_bike      1181
scooter            1153
mechanical_bike    1146
e-bike             1129
TROTTINETTE        1118
E-BIKE             1068
Name: count, dtype: int64

--- Vehicle type standardization ---
Affected rows: 2249
Action: Standardized recognized labels; unknown labels flagged
Reason: Only three canonical vehicle types are allowed
Rows remaining: 6795

Standardize